# 2変数の既知勾配：weighted ridge fit と中心差分の比較

HFSS の Monte Carlo susceptibility notebook で使っている **Gaussian-weighted ridge fit** と、各軸を正負に動かす **central difference（中心差分）** を、解析勾配が既知の2変数関数で比較します。

この notebook はリポジトリ固有モジュールや HFSS を必要とせず、NumPy / pandas / matplotlib だけで独立実行できます。

## 比較する量

中心 $x_0$ における真の勾配 $g=\nabla f(x_0)$ に対して、各推定値 $\hat g$ の

- 絶対誤差：$\|\hat g-g\|_2$
- 相対誤差：$\|\hat g-g\|_2/\|g\|_2$
- 関数評価回数

を比較します。

### 評価回数

- weighted ridge fit：中心1回 + Monte Carlo摂動点 $M$ 回、合計 **$M+1$ 回**
- 2変数の中心差分：各変数の $+h_i,-h_i$、合計 **$2d=4$ 回**（中心値は不要）
- noisy central difference：各側を $R$ 回反復すると **$4R$ 回**

既に中心点のHFSS結果がある場合、fitの追加評価回数は $M$ 回と読み替えられます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)
pd.options.display.float_format = "{:.6g}".format


## 1. 解析勾配が既知のテスト関数

非線形性と変数間相互作用を含む滑らかな関数を使います。

$$
f(x_1,x_2)=\sin(x_1)+0.3x_1x_2+0.5x_2^2+0.05x_1^3
$$

$$
\nabla f=\left(\cos(x_1)+0.3x_2+0.15x_1^2,\;0.3x_1+x_2\right)
$$

中心は $x_0=(0.4,-0.3)$、Monte Carlo摂動標準偏差は $\sigma=(0.15,0.10)$ とします。


In [ ]:
def objective(x):
    """Vectorized scalar objective; the last axis is (x1, x2)."""
    x = np.asarray(x, dtype=float)
    x1, x2 = x[..., 0], x[..., 1]
    return np.sin(x1) + 0.3 * x1 * x2 + 0.5 * x2**2 + 0.05 * x1**3


def analytic_gradient(x):
    x = np.asarray(x, dtype=float)
    x1, x2 = x
    return np.array([
        np.cos(x1) + 0.3 * x2 + 0.15 * x1**2,
        0.3 * x1 + x2,
    ])


x0 = np.array([0.4, -0.3])
sigma = np.array([0.15, 0.10])
g_true = analytic_gradient(x0)
print("x0             =", x0)
print("sigma          =", sigma)
print("analytic grad  =", g_true)


## 2. 推定器

### Gaussian-weighted ridge fit

Monte Carlo点 $x_k=x_0+\delta_k$ に対し、

$$
f(x_k)-f(x_0)\approx\beta^T\delta_k
$$

をfitします。実際のvalidation notebookと同じく、$z_{k,i}=\delta_{k,i}/\sigma_i$ で標準化し、

$$w_k=\exp(-\|z_k\|^2/2)$$

を使います。`ridge=1e-10` は実質的には特異行列回避用の微小jitterです。

### 中心差分

$$
\frac{\partial f}{\partial x_i}\approx
\frac{f(x_0+h_i e_i)-f(x_0-h_i e_i)}{2h_i}
$$


In [ ]:
def weighted_ridge_gradient(
    func, x_center, sigma_vec, n_samples, rng, ridge=1e-10,
    gaussian_weights=True, noise_std=0.0,
):
    """Estimate a local gradient from simultaneous Gaussian perturbations."""
    x_center = np.asarray(x_center, dtype=float)
    sigma_vec = np.asarray(sigma_vec, dtype=float)
    z = rng.normal(size=(int(n_samples), x_center.size))
    dx = z * sigma_vec
    X = x_center + dx

    # Count one center evaluation plus n_samples perturbed evaluations.
    y0 = float(func(x_center)) + rng.normal(scale=noise_std)
    y = np.asarray(func(X), dtype=float) + rng.normal(scale=noise_std, size=n_samples)
    dy = y - y0

    weights = np.exp(-0.5 * np.sum(z**2, axis=1)) if gaussian_weights else np.ones(n_samples)
    sqrt_w = np.sqrt(weights)
    A = z * sqrt_w[:, None]
    b = dy * sqrt_w
    beta_scaled = np.linalg.solve(A.T @ A + ridge * np.eye(x_center.size), A.T @ b)
    gradient = beta_scaled / sigma_vec
    return gradient, n_samples + 1


def central_difference_gradient(func, x_center, step, noise_std=0.0, repeats=1, rng=None):
    """Central difference, optionally averaging repeated noisy evaluations per side."""
    x_center = np.asarray(x_center, dtype=float)
    step = np.broadcast_to(np.asarray(step, dtype=float), x_center.shape)
    rng = np.random.default_rng() if rng is None else rng
    gradient = np.empty_like(x_center)
    for i, h in enumerate(step):
        direction = np.zeros_like(x_center)
        direction[i] = h
        y_plus = np.asarray([func(x_center + direction) for _ in range(repeats)], dtype=float)
        y_minus = np.asarray([func(x_center - direction) for _ in range(repeats)], dtype=float)
        if noise_std:
            y_plus += rng.normal(scale=noise_std, size=repeats)
            y_minus += rng.normal(scale=noise_std, size=repeats)
        gradient[i] = (y_plus.mean() - y_minus.mean()) / (2.0 * h)
    return gradient, 2 * x_center.size * repeats


def error_metrics(estimate, truth):
    absolute = float(np.linalg.norm(np.asarray(estimate) - truth))
    relative = absolute / float(np.linalg.norm(truth))
    return absolute, relative


## 3. 1回のデモ

fitは現在のMC notebookと同じ100摂動点、中心差分は4回の評価で比較します。中心差分のstepはここでは $0.5\sigma$ とします。


In [ ]:
rng = np.random.default_rng(2026)
g_fit, calls_fit = weighted_ridge_gradient(objective, x0, sigma, 100, rng)
g_cd, calls_cd = central_difference_gradient(objective, x0, 0.5 * sigma)

rows = []
for name, estimate, calls in [
    ("weighted ridge fit (M=100)", g_fit, calls_fit),
    ("central difference (h=0.5 sigma)", g_cd, calls_cd),
]:
    abs_err, rel_err = error_metrics(estimate, g_true)
    rows.append({
        "method": name,
        "evaluations": calls,
        "grad_x1": estimate[0],
        "grad_x2": estimate[1],
        "L2_error": abs_err,
        "relative_error": rel_err,
    })

single_run_df = pd.DataFrame(rows)
display(single_run_df)


## 4. ノイズなし：計算回数と精度

weighted fitには乱数ばらつきがあるため、各 $M$ について500 seedで反復し、誤差の中央値と10–90 percentileを集計します。中心差分はstep幅を変えて、打ち切り誤差を確認します。


In [ ]:
def summarize(values):
    values = np.asarray(values, dtype=float)
    return {
        "median_L2_error": np.median(values),
        "p10_L2_error": np.percentile(values, 10),
        "p90_L2_error": np.percentile(values, 90),
    }


n_trials = 500
fit_rows = []
for m in [4, 8, 16, 32, 64, 100, 200]:
    errors = []
    for seed in range(n_trials):
        estimate, calls = weighted_ridge_gradient(
            objective, x0, sigma, m, np.random.default_rng(seed)
        )
        errors.append(error_metrics(estimate, g_true)[0])
    fit_rows.append({"method": "weighted ridge", "M_or_step_scale": m, "evaluations": calls, **summarize(errors)})

cd_rows = []
for step_scale in [2.0, 1.0, 0.5, 0.25, 0.1, 0.01, 0.001]:
    estimate, calls = central_difference_gradient(objective, x0, step_scale * sigma)
    error = error_metrics(estimate, g_true)[0]
    cd_rows.append({
        "method": "central difference",
        "M_or_step_scale": step_scale,
        "evaluations": calls,
        "median_L2_error": error,
        "p10_L2_error": error,
        "p90_L2_error": error,
    })

fit_convergence_df = pd.DataFrame(fit_rows)
cd_step_df = pd.DataFrame(cd_rows)
print("Weighted fit convergence (500 random designs per M)")
display(fit_convergence_df)
print("Central-difference step study")
display(cd_step_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x = fit_convergence_df["evaluations"].to_numpy()
y = fit_convergence_df["median_L2_error"].to_numpy()
lo = fit_convergence_df["p10_L2_error"].to_numpy()
hi = fit_convergence_df["p90_L2_error"].to_numpy()
ax.plot(x, y, "o-", label="weighted fit median")
ax.fill_between(x, lo, hi, alpha=0.2, label="10–90 percentile")
ax.set(xlabel="function evaluations", ylabel="L2 gradient error", title="Weighted fit: budget vs accuracy")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

ax = axes[1]
ax.loglog(cd_step_df["M_or_step_scale"], cd_step_df["median_L2_error"], "o-")
ax.set(xlabel="step / sigma", ylabel="L2 gradient error", title="Central difference: step sensitivity (4 evaluations)")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


## 5. HFSSの数値揺らぎを模した比較

各関数評価に独立なGaussianノイズを加えます。ここでは `noise_std = 1e-3` とします。

- fit：budget $B$ に対し、中心1回 + $B-1$ Monte Carlo点
- 中心差分：各 $+h/-h$ を可能な回数だけ反復平均し、合計 $4R\le B$

中心差分のstepは $0.5\sigma$ 固定です。ノイズ下ではstepを小さくしすぎると、差分でノイズが増幅される点に注意してください。


In [ ]:
noise_std = 1e-3
n_trials_noise = 500
noisy_rows = []

for budget in [8, 16, 32, 64, 100]:
    fit_errors = []
    cd_errors = []
    fit_calls = None
    cd_calls = None
    repeats = max(1, budget // 4)
    for seed in range(n_trials_noise):
        g_fit_noisy, fit_calls = weighted_ridge_gradient(
            objective, x0, sigma, budget - 1, np.random.default_rng(seed), noise_std=noise_std
        )
        g_cd_noisy, cd_calls = central_difference_gradient(
            objective, x0, 0.5 * sigma, noise_std=noise_std,
            repeats=repeats, rng=np.random.default_rng(100_000 + seed),
        )
        fit_errors.append(error_metrics(g_fit_noisy, g_true)[0])
        cd_errors.append(error_metrics(g_cd_noisy, g_true)[0])

    noisy_rows.append({
        "method": "weighted ridge",
        "requested_budget": budget,
        "evaluations": fit_calls,
        **summarize(fit_errors),
    })
    noisy_rows.append({
        "method": "central difference",
        "requested_budget": budget,
        "evaluations": cd_calls,
        **summarize(cd_errors),
    })

noisy_df = pd.DataFrame(noisy_rows)
display(noisy_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for method, group in noisy_df.groupby("method", sort=False):
    x = group["evaluations"].to_numpy()
    y = group["median_L2_error"].to_numpy()
    lo = group["p10_L2_error"].to_numpy()
    hi = group["p90_L2_error"].to_numpy()
    ax.plot(x, y, "o-", label=method)
    ax.fill_between(x, lo, hi, alpha=0.15)
ax.set(xlabel="function evaluations", ylabel="L2 gradient error", title=f"Noisy evaluations (noise std={noise_std:g})")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 6. 読み方と実務上の判断

1. **ノイズなし・純粋な中心勾配が目的**なら、2変数で4回だけの中心差分が通常は圧倒的に効率的です。
2. **中心差分はstep依存**です。大きすぎると非線形性、小さすぎるとHFSSの数値揺らぎや丸め誤差を増幅します。
3. **weighted fitは乱数設計に依存**し、中心1回を含めて $M+1$ 回必要ですが、多数点で平均するためノイズに対して頑健になり得ます。
4. fitの傾きは厳密な一点微分ではなく、指定した摂動範囲における局所best-fit slopeです。
5. Monte Carlo点は既にGaussian分布から生成されています。さらにGaussian weightを掛けると中心を二重に強調します。製造ばらつき分布全体のbest-fit slopeが目的なら、`gaussian_weights=False` も比較してください。
6. HFSSでは、まず `h = 0.25σ, 0.5σ, 1.0σ` の中心差分を比較し、左右差分の整合性を確認するのが安全です。その後、MC fitと一致するかを見ると、非線形性・交互作用・数値ノイズを診断できます。

このデモの数値はセルを実行した環境で生成されます。関数、中心、sigma、ノイズ強度を変更して再実行できます。
